In [1]:
import os
import sys
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.model_selection import train_test_split

import scanpy as sc
import anndata

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
)

from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
import evaluate

sys.path.append("..")
from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

/data/xcx/anaconda3/envs/Llama/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Load the pre-trained cell tokenizer

In [2]:
import warnings
warnings.filterwarnings("ignore")
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4) # 1e-3
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# -- device
if torch.cuda.is_available():
    args.device='cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device='cpu' 
print(args.device)
    
data_dim = 5000
vae = VQVAE(data_dim, args.n, args.codebook_dim, args.n_codebooks, args.encoder_hidden_dim, args.decoder_hidden_dim, args.beta, args.gamma, args.dropout_rate).to(args.device)

vae.load_state_dict(torch.load('../model/cell_tokenzier/model.pth')['model'])
vae.eval() 

cuda


VQVAE(
  (encoder): Encoder(
    (network): Sequential(
      (0): Linear(in_features=5000, out_features=1600, bias=True)
      (1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (2): PReLU(num_parameters=1)
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=1600, out_features=1024, bias=True)
      (5): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (6): PReLU(num_parameters=1)
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=1024, out_features=800, bias=True)
      (9): LayerNorm((800,), eps=1e-05, elementwise_affine=True)
      (10): PReLU(num_parameters=1)
      (11): Dropout(p=0.2, inplace=False)
      (12): Linear(in_features=800, out_features=256, bias=True)
    )
  )
  (vector_quantization): VectorQuantizer(
    (embedding): Embedding(256, 8)
  )
  (decoder): Decoder(
    (network): Sequential(
      (0): Linear(in_features=256, out_features=800, bias=True)
      (1): LayerNorm((800,), eps=1e-05, elementwise

#### Prepare Model and Tokenizer
Now, we will be adding 258 new tokens as well as replace the existing pad, bos and eos tokens of the model.

In [3]:
# Define new tokens
new_tokens = [f'<code_{i}>' for i in range(256)] + ['<cell_token_start>', '<cell_token_end>']

In [ ]:
model_name = "your path to the LLM backbone"  # load the base LLM backbone
tokenizer_path = "your path to LLM's tokenizer"  # load the LLM toknizer

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_path,
    pad_token="<|pad|>",
    eos_token="<|endsentence|>",
    additional_special_tokens=new_tokens,
)

In [ ]:
len(tokenizer)

#### Load the processed dataset.

In [1]:
from datasets import load_dataset, load_from_disk
from tqdm import tqdm

max_length = 100
batch_size = 128

processed_datasets = load_from_disk("../data/blood/tokenized_dataset_annotation/")

random select 1000 samples for test

In [8]:
eval_subset = processed_datasets
eval_subset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

eval_subset = eval_subset.shuffle(seed=42).select(range(1000))
eval_subset

Dataset({
    features: ['input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

#### Load the pre-trained CellTok model

In [9]:
from peft import PeftModel

inference_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
)
inference_model.resize_token_embeddings(len(tokenizer))

inference_model = PeftModel.from_pretrained(inference_model, "your path to CellTok-base")

inference_model.to("cuda")
inference_model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

PeftModel(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): lora.Embedding(
          (base_layer): Embedding(151925, 3584)
          (lora_dropout): ModuleDict(
            (default): Identity()
          )
          (lora_A): ModuleDict()
          (lora_B): ModuleDict()
          (lora_embedding_A): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 64x151925 (cuda:0)])
          (lora_embedding_B): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 3584x64 (cuda:0)])
          (lora_magnitude_vector): ModuleDict()
        )
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
     

#### Evaluation

Extract the predicted/target celltype from text.

In [10]:
import re

def extract_cell_type(text, is_target=False):
    from_pattern = r'this cell is (?:a|an) (.*?) from'
    
    dot_pattern = r'this cell is (?:a|an) (.*?)\.'
    
    if is_target:
        match = re.search(from_pattern, text, re.IGNORECASE)
    else:
        match = re.search(from_pattern, text, re.IGNORECASE)
        if not match:
            match = re.search(dot_pattern, text, re.IGNORECASE)
    
    if match:
        return match.group(1).strip()
    return ""

def evaluate_cell_type_and_disease(target, target_predicted):
    target_cell_type = extract_cell_type(target, is_target=False)
    predicted_cell_type = extract_cell_type(target_predicted, is_target=False)
    cell_type_correct = (target_cell_type == predicted_cell_type)

    return {
        "cell_type_correct": cell_type_correct,
        "target_cell_type": target_cell_type,
        "predicted_cell_type": predicted_cell_type,
    }

In [12]:
from tqdm import tqdm

results = []
results_original = []

for sample in tqdm(eval_subset, desc="Evaluating", ncols=100):  
    sample = {k: v.unsqueeze(0) for k, v in sample.items()}  
    sample = {k: v.to("cuda") for k, v in sample.items()}
    outputs = inference_model(**sample)
    
    valid_label_positions = (sample['labels'] == -100) & (sample['attention_mask'] == 1)
    filtered_input_ids = sample['input_ids'][valid_label_positions]
    output_tokens = inference_model.generate(
        filtered_input_ids.unsqueeze(0),
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        top_k=50,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    context = tokenizer.decode(filtered_input_ids, skip_special_tokens=False)

    query = "This cell is an "

    target_predicted = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    results_original.append(target_predicted)

    labels = sample["labels"][0].cpu().tolist()
    valid_labels = [token for token in labels if token != -100]
    target = tokenizer.decode(valid_labels, skip_special_tokens=False)

    eval_result = evaluate_cell_type_and_disease(target, target_predicted)
    results.append(eval_result)

cell_type_correct = sum([result["cell_type_correct"] for result in results]) / len(results)

print(f"Cell Type Accuracy: {cell_type_correct:.2f}")

Evaluating: 100%|█████████████████████████████████████████████████| 100/100 [03:08<00:00,  1.88s/it]

Cell Type Accuracy: 0.11
